In [ ]:
# ⚠️  OLD CELL — superseded. Run cell 2117b490 below instead.
# (Paths here point to deleted data — this cell will raise if run.)
raise RuntimeError("Run the updated training cell (2117b490) below — not this one.")

In [ ]:
# ── Show final/peak validation Dice for the latest v3 run ─────────────────────
from pathlib import Path
import os, csv, json, re

# Point to your v3 root
RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/Low_Quality_Train/Low_Quality_Train")

def latest_run_dir(run_root: Path) -> Path:
    runs_dir = run_root / "runs"
    candidates = [p for p in runs_dir.glob("*") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No run folders found under {runs_dir}")
    # pick most recently modified run folder
    return max(candidates, key=lambda p: p.stat().st_mtime)

def read_from_history_csv(cb_dir: Path):
    csv_path = cb_dir / "history.csv"
    if not csv_path.exists():
        return None
    best_val = best_epoch = last_val = last_epoch = None
    with open(csv_path, newline="") as f:
        for i, row in enumerate(csv.DictReader(f)):
            v = row.get("val_dice_coefficient")
            if not v:
                continue
            val = float(v)
            last_val, last_epoch = val, i
            if best_val is None or val > best_val:
                best_val, best_epoch = val, i
    if last_val is None:
        return None
    return {
        "source": "CSV",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(csv_path),
    }

def read_from_history_json(cb_dir: Path):
    jpath = cb_dir / "artifacts" / "history_epoch.json"
    if not jpath.exists():
        return None
    with open(jpath) as f:
        h = json.load(f)
    vals = h.get("val_dice_coefficient") or []
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "JSON",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(jpath),
    }

def read_from_log(log_dir: Path):
    log_path = log_dir / "train_stdout_stderr.log"
    if not log_path.exists():
        return None
    pat = re.compile(r"val_dice_coefficient:\s*([0-9]*\.?[0-9]+)")
    vals = []
    with open(log_path, "r", errors="ignore") as f:
        for line in f:
            m = pat.search(line)
            if m:
                vals.append(float(m.group(1)))
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "LOG",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(log_path),
    }

# Resolve the latest run under v3
run_dir = latest_run_dir(RUN_ROOT)
cb_dir  = run_dir / "callbacks"
log_dir = run_dir / "logs"

# Prefer CSV → JSON → logs
res = (read_from_history_csv(cb_dir)
       or read_from_history_json(cb_dir)
       or read_from_log(log_dir))

print(f"Using RUN_DIR: {run_dir}")
if res:
    print(f"[{res['source']}] last val_dice: {res['last_val']:.6f} (epoch {res['last_epoch']})")
    print(f"[{res['source']}] best  val_dice: {res['best_val']:.6f} (epoch {res['best_epoch']})")
    print(f"Source file: {res['path']}")
else:
    print("Could not find val_dice in CSV, JSON, or logs for this run.")


# Train v3.1 resume v3 training

In [ ]:
# === ARC_ATLAS_Train_v3_Low_Quality — train on lowest-quality 522 non-held-out cases ===
from pathlib import Path
import importlib.util, os, sys, gc, time, traceback, shlex, subprocess
import tensorflow as tf
from tensorflow.keras import mixed_precision

# --------- Paths ----------
CUDA_ID = "0"
SPLIT_ROOT  = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_LowQualityTrain_Split_Data/train_low_quality").parent
TRAIN_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_LowQualityTrain_Split_Data/train_low_quality")
TRAIN_T1    = TRAIN_DIR / "t1"      # <- use separated subfolder ONLY
TRAIN_MASKS = TRAIN_DIR / "masks"   # <- use separated subfolder ONLY
SPLIT_SUMMARY = SPLIT_ROOT / "split_summary.json"

RUN_ROOT   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/Low_Quality_Train/Low_Quality_Train")
MODULE_PATH = Path("/home/rbielski/stroke_cleaned/stroke_segmentation_v1.2/stroke_seg_v1.2_train.py")

# --------- New run folders ----------
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = RUN_ROOT / "runs" / RUN_ID
MODEL_DIR = RUN_DIR / "models"
CALLBACKS_DIR = RUN_DIR / "callbacks"
LOG_DIR = RUN_DIR / "logs"
for d in (MODEL_DIR, CALLBACKS_DIR, LOG_DIR): d.mkdir(parents=True, exist_ok=True)

# --------- Env & TF init ----------
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_ID
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["SMARTSOTA_LOG_DIR"] = str(LOG_DIR)

tf.keras.backend.clear_session(); gc.collect()
mixed_precision.set_global_policy("mixed_float16")

# Optional: tee logs to file and console
class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, data): 
        for s in self.streams: s.write(data); s.flush()
        return len(data)
    def flush(self): 
        for s in self.streams: s.flush()
log_file = open(LOG_DIR / "train_stdout_stderr.log", "a", buffering=1)
sys.stdout = Tee(sys.__stdout__, log_file)
sys.stderr = Tee(sys.__stderr__, log_file)

print("Run ID:", RUN_ID)
print("TF:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
print(f"Train images: {len(list(TRAIN_T1.glob('*.nii.gz')))}  masks: {len(list(TRAIN_MASKS.glob('*.nii.gz')))}")
print("Split summary:", SPLIT_SUMMARY)

print("Split summary:", SPLIT_SUMMARY)

for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception as e: print("set_memory_growth failed:", e)

# --------- Import training module; avoid MirroredStrategy on 1 GPU ----------
spec = importlib.util.spec_from_file_location("arc_seg_train", MODULE_PATH)
seg = importlib.util.module_from_spec(spec)
seg.tf = tf
spec.loader.exec_module(seg)

# Force default (no mirrored). Saves VRAM and matches earlier good runs.
seg.strategy = tf.distribute.get_strategy()
print("Strategy:", type(seg.strategy).__name__)

# --------- Hyperparams (identical to successful Nov 2025 run) ----------
INPUT_SHAPE   = (192, 224, 192, 1)
BATCH_SIZE    = 1
BASE_FILTERS  = 8
SAM_HEADS     = 2
AUG_INTENSITY = 0.30
VAL_SPLIT     = 0.15
TOTAL_EPOCHS  = 140
INITIAL_EPOCH = 0

# LR schedule (the one that worked)
INITIAL_LR   = 1e-4
MIN_LR       = 5e-7
WARMUP_EPOCHS= 15

# --------- Launch training (FRESH: no resume, no load) ----------
try:
    history = seg.train_dynamic_model(
        DATA_DIR=TRAIN_DIR,
        IMAGES_DIR=TRAIN_T1,
        MASKS_DIR=TRAIN_MASKS,

        MODEL_DIR=MODEL_DIR,
        CALLBACKS_DIR=CALLBACKS_DIR,

        TOTAL_EPOCHS=TOTAL_EPOCHS,
        INITIAL_EPOCH=INITIAL_EPOCH,
        LOAD_WEIGHTS_FROM=None,
        RESUME_FROM_LATEST=False,

        INPUT_SHAPE=INPUT_SHAPE,
        BATCH_SIZE=BATCH_SIZE,
        BASE_FILTERS=BASE_FILTERS,
        SAM_HEADS=SAM_HEADS,
        RESAMPLE_TO_TARGET=True,

        AUGMENTATION_INTENSITY=AUG_INTENSITY,
        VALIDATION_SPLIT=VAL_SPLIT,

        INITIAL_LR=INITIAL_LR,
        MIN_LR=MIN_LR,
        WARMUP_EPOCHS=WARMUP_EPOCHS,
    )
    print("Training complete. Logged keys:", list(getattr(history, "history", {}).keys()))
    print("Run artifacts at:", RUN_DIR)

except Exception as e:
    print("\n================= UNCAUGHT EXCEPTION =================")
    traceback.print_exc()
    print("======================================================\n")
    try:
        print("Last few GPU snapshots:")
        for _ in range(3):
            subprocess.run(shlex.split("nvidia-smi"), check=False)
            time.sleep(1)
    except Exception:
        pass
    raise
finally:
    try: log_file.flush()
    except Exception: pass


In [ ]:
# ── Show final/peak validation Dice for the latest v3 run ─────────────────────
from pathlib import Path
import os, csv, json, re

# Point to your v3 root
RUN_ROOT = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v3/Low_Quality_Train/Low_Quality_Train")

def latest_run_dir(run_root: Path) -> Path:
    runs_dir = run_root / "runs"
    candidates = [p for p in runs_dir.glob("*") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError(f"No run folders found under {runs_dir}")
    # pick most recently modified run folder
    return max(candidates, key=lambda p: p.stat().st_mtime)

def read_from_history_csv(cb_dir: Path):
    csv_path = cb_dir / "history.csv"
    if not csv_path.exists():
        return None
    best_val = best_epoch = last_val = last_epoch = None
    with open(csv_path, newline="") as f:
        for i, row in enumerate(csv.DictReader(f)):
            v = row.get("val_dice_coefficient")
            if not v:
                continue
            val = float(v)
            last_val, last_epoch = val, i
            if best_val is None or val > best_val:
                best_val, best_epoch = val, i
    if last_val is None:
        return None
    return {
        "source": "CSV",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(csv_path),
    }

def read_from_history_json(cb_dir: Path):
    jpath = cb_dir / "artifacts" / "history_epoch.json"
    if not jpath.exists():
        return None
    with open(jpath) as f:
        h = json.load(f)
    vals = h.get("val_dice_coefficient") or []
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "JSON",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(jpath),
    }

def read_from_log(log_dir: Path):
    log_path = log_dir / "train_stdout_stderr.log"
    if not log_path.exists():
        return None
    pat = re.compile(r"val_dice_coefficient:\s*([0-9]*\.?[0-9]+)")
    vals = []
    with open(log_path, "r", errors="ignore") as f:
        for line in f:
            m = pat.search(line)
            if m:
                vals.append(float(m.group(1)))
    if not vals:
        return None
    best_val = max(vals)
    best_epoch = vals.index(best_val)
    last_val = vals[-1]
    last_epoch = len(vals) - 1
    return {
        "source": "LOG",
        "last_val": last_val, "last_epoch": last_epoch,
        "best_val": best_val, "best_epoch": best_epoch,
        "path": str(log_path),
    }

# Resolve the latest run under v3
run_dir = latest_run_dir(RUN_ROOT)
cb_dir  = run_dir / "callbacks"
log_dir = run_dir / "logs"

# Prefer CSV → JSON → logs
res = (read_from_history_csv(cb_dir)
       or read_from_history_json(cb_dir)
       or read_from_log(log_dir))

print(f"Using RUN_DIR: {run_dir}")
if res:
    print(f"[{res['source']}] last val_dice: {res['last_val']:.6f} (epoch {res['last_epoch']})")
    print(f"[{res['source']}] best  val_dice: {res['best_val']:.6f} (epoch {res['best_epoch']})")
    print(f"Source file: {res['path']}")
else:
    print("Could not find val_dice in CSV, JSON, or logs for this run.")
